# Validación en producción · figuras del anexo

TFM Cercanías RENFE · UCM · 2026

Regenera las figuras del anexo. **Ninguna cifra está escrita a mano**: cada figura
declara su fuente de datos en la celda de texto que la precede.

| Figura | Fuente |
|---|---|
| 1, 2, 3, 5 | `verificacion.csv` |
| 4 | `medicion_rumbo.csv`, que genera `medir_rumbo.py` en el VPS |
| 6 | `verificacion_15sep.csv`, validación tras el filtro de dominio |

`verificacion.csv` contiene las 613 predicciones que la aplicación sirvió el
13/09/2026, contrastadas al día siguiente con la llegada real de cada tren.

Las figuras se guardan como PNG junto al cuaderno.


## Preparación

Se trabaja solo con las 548 predicciones de la vía directa: las 65 restantes proceden
de una medida indirecta que no es comparable.


In [ ]:
import matplotlib
import logging
import matplotlib.pyplot as plt

# Sin Arial en el sistema, matplotlib avisa en cada texto; se usa la siguiente fuente de la lista.
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
import matplotlib.ticker as mticker
import numpy as np, pandas as pd

TINTA, ACENTO, GRIS = "#101826", "#e8501e", "#8a94a3"
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 220, "savefig.bbox": "tight",
                     "font.family": ["Arial", "Liberation Sans", "DejaVu Sans"], "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})


# --- Coma decimal ------------------------------------------------------------
# La memoria usa coma decimal. FMT se aplica a los ejes numéricos (no a los de
# categorías ni a los de fechas) y coma() a toda etiqueta con cifras.
def coma(valor, decimales=1):
    return f"{valor:.{decimales}f}".replace(".", ",").replace("-", "−")

FMT = mticker.FuncFormatter(lambda v, _: f"{v:g}".replace(".", ",").replace("-", "−"))

df = pd.read_csv("verificacion.csv")
df = df[df["via"] == "parada"].copy()
df["pred_min"] = df["retraso_predicho_s"] / 60
df["real_min"] = df["retraso_real_s"] / 60
df["err_min"] = df["pred_min"] - df["real_min"]
print("n =", len(df))

## 1. La compresión del rango

**Datos:** `verificacion.csv`, columnas `retraso_predicho_s` y `retraso_real_s`.

Es el hallazgo central. El modelo casi nunca predice los retrasos grandes: la
desviación típica del retraso real es 1,9 veces la del predicho, y llega a 3,4 veces
si se excluyen las 28 predicciones negativas, que son valores extremos del propio
modelo.

**Para la presentación:** una diapositiva con esta figura y la frase «el modelo
acierta el nivel medio, no distingue qué tren llegará tarde».


In [ ]:
# --- F1: compresion del rango -------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 4.2))
bins = np.arange(-4, 28, 1)
ax.hist(df["real_min"], bins=bins, color=ACENTO, alpha=.55, label="observado")
ax.hist(df["pred_min"], bins=bins, color=TINTA, alpha=.65, label="predicho")
ax.axvline(df["real_min"].std(), lw=0, label=f"desv. típica observada {coma(df['real_min'].std())} min")
ax.axvline(df["pred_min"].std(), lw=0, label=f"desv. típica predicha {coma(df['pred_min'].std())} min")
ax.set_xlabel("Retraso (minutos)"); ax.set_ylabel("Número de predicciones")
ax.set_title("El modelo comprime el rango: nunca predice los retrasos grandes", loc="left")
ax.legend(frameon=False)
ax.xaxis.set_major_formatter(FMT)
fig.tight_layout(); fig.savefig("f1_compresion.png"); plt.close(fig)

## 2. El modelo frente a predictores constantes

**Datos:** el mismo CSV. Las constantes se evalúan sobre las mismas 548 observaciones.

Las constantes se eligen conociendo el resultado del día, así que juegan con ventaja.
El punto no es quién gana por unas décimas, sino que el modelo explica el 7 % de la
variación del retraso real.

**Para la presentación:** llévala solo si vas a defender el diagnóstico. Es la figura
más dura del conjunto y, contada con seguridad, la que más criterio demuestra.


In [ ]:
# --- F2: el modelo frente a constantes ---------------------------------------
mae = lambda p: float(np.abs(p - df["real_min"]).mean())
etiquetas = ["Modelo\nLightGBM", "Constante\n4 min", "Constante\n6,45 min", "Constante\n0 min"]
valores = [mae(df["pred_min"]), mae(4.0), mae(6.45), mae(0.0)]
fig, ax = plt.subplots(figsize=(7, 4))
colores = [ACENTO] + [GRIS] * 3
b = ax.bar(etiquetas, valores, color=colores)
ax.bar_label(b, labels=[f"{coma(v, 2)} min" for v in valores], padding=3, fontweight="bold")
ax.set_ylabel("Error absoluto medio (minutos)"); ax.set_ylim(0, max(valores) * 1.2)
ax.yaxis.set_major_formatter(FMT)
ax.set_title("Una constante bien elegida iguala al modelo", loc="left")
fig.tight_layout(); fig.savefig("f2_baseline.png"); plt.close(fig)

## 3. Sesgo y correlación según el horizonte

**Datos:** el mismo CSV, agrupando por `horizonte_min`.

Justifica el filtro de dominio. El sesgo se agrava al alejarse la llegada, pero la
correlación no cae: el problema no es la distancia hasta la llegada, sino la
antelación respecto a la salida del tren.

Los tramos extremos se excluyen por tamaño de muestra: 15 observaciones por debajo de
45 minutos y 9 por encima de 180. Con ellos dentro el sesgo deja de ser monótono, por
ruido muestral y no por comportamiento del modelo. Está declarado en el código y debe
declararse también al presentarla.


In [ ]:
# --- F3: sesgo y correlacion por horizonte ------------------------------------
# Se excluyen los extremos por tamano de muestra: por debajo de 45 min solo hay 15
# observaciones y por encima de 180 solo 9. Con esos dos dentro, el sesgo deja de
# ser monotono por ruido muestral, no por comportamiento del modelo. Declararlo es
# mejor que elegir los cortes que dibujan la conclusion que uno quiere.
cortes = [45, 60, 90, 120, 180]
nombres = ["45-60", "60-90", "90-120", "120-180"]
sub = df[(df["horizonte_min"] > 45) & (df["horizonte_min"] <= 180)].copy()
sub["tramo"] = pd.cut(sub["horizonte_min"], cortes, labels=nombres)
g = sub.groupby("tramo", observed=True)
sesgo = g["err_min"].mean().reindex(nombres)
corr = g[["pred_min", "real_min"]].corr().unstack().iloc[:, 1].reindex(nombres)
n = g.size().reindex(nombres)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
b = a1.bar(nombres, sesgo, color=ACENTO)
a1.bar_label(b, labels=[coma(v) for v in sesgo], padding=-16, color="white", fontweight="bold")
a1.axhline(0, color=TINTA, lw=1)
a1.set_ylabel("Sesgo: predicho − observado (min)")
a1.set_title("El sesgo se agrava con el horizonte", loc="left")

b = a2.bar(nombres, corr, color=TINTA)
a2.bar_label(b, labels=[coma(v, 2) for v in corr], padding=3, fontweight="bold")
a2.axhline(0, color=GRIS, lw=1)
a2.set_ylabel("Correlación predicho-observado")
# Holgura por abajo: la etiqueta de la barra negativa quedaba sobre el eje.
a2.set_ylim(min(-0.1, float(corr.min()) * 2.2), max(0.5, float(corr.max()) * 1.35))
# Afirmacion NEGATIVA a proposito: la correlacion no es monotona (0,26 - 0,08 - 0,41)
# y seria deshonesto titular que mejora. Lo que el dato sostiene es que NO empeora,
# y eso basta para el argumento: el problema no esta en la distancia a la llegada.
a2.set_title("La correlación no cae con él", loc="left")

for ax in (a1, a2):
    ax.set_xlabel("Horizonte hasta la llegada (minutos)")
    ax.set_xticks(range(len(nombres)))
    ax.set_xticklabels([f"{x}\nn={n[x]}" for x in nombres], fontsize=9)
    ax.yaxis.set_major_formatter(FMT)
fig.tight_layout()
# Nota al pie DESPUÉS de tight_layout y guardada con bbox tight: antes quedaba cortada.
fig.text(0.01, -0.02, "Horizontes de 45 a 180 min · los extremos se excluyen por tamaño de muestra",
         fontsize=9, color=GRIS)
fig.savefig("f3_horizonte.png"); plt.close(fig)

## 4. El rumbo derivado, contrastado contra el terreno

**Datos:** `medicion_rumbo.csv`, que produce `medir_rumbo.py` **ejecutándose en el
VPS**. No salen de `verificacion.csv`, así que hay que generarlo antes:

```bash
# [VPS]
cd /home/tfm/renfe-delay-app/api
sudo -u tfm /usr/bin/python3 ../scripts/medir_rumbo.py
```

El script empareja dos capturas consecutivas del feed por identificador de trayecto,
filtra al núcleo de Madrid y compara el rumbo de cada regla candidata con el rumbo
real del desplazamiento del tren entre ambas capturas. El desplazamiento observado es
independiente del catálogo, que es justamente lo que se quiere validar.

**Si el fichero no está, esta celda falla.** Es deliberado: antes ninguna figura que
una figura con números escritos a mano.

**Para la presentación:** es la mejor historia metodológica del proyecto. Derivar una
magnitud que la fuente no publica, someterla a prueba contra el terreno y rehacerla
cuando no la supera.


In [ ]:
# --- F4: rumbo derivado frente al desplazamiento real -------------------------
# Los datos los produce medir_rumbo.py EN EL VPS, que empareja dos capturas
# consecutivas del feed por tripId y compara cada regla candidata con el rumbo real
# del desplazamiento del tren. Si el fichero no esta, esta celda falla a proposito:
# antes ninguna figura que dibujar que una figura con numeros escritos a mano.
rumbo = pd.read_csv("medicion_rumbo.csv")
reglas = [("desv_publicado", "Desde el tren\nal stopId publicado"),
          ("desv_siguiente", "Desde el tren\na la parada siguiente"),
          ("desv_tramo",     "Rumbo del TRAMO de vía\n(regla implantada)")]
med = [(et, rumbo[c].dropna().median(), int(rumbo[c].notna().sum()),
        int((rumbo[c] > 135).sum())) for c, et in reglas]

fig, ax = plt.subplots(figsize=(8.5, 4))
b = ax.barh([m[0] for m in med], [m[1] for m in med], color=[GRIS, GRIS, ACENTO])
etiquetas = ax.bar_label(b, labels=[f"{coma(m[1])}°   n = {m[2]} · {m[3]} invertidos" for m in med],
                         padding=6, fontsize=9, fontweight="bold")
for t in etiquetas:   # fondo blanco: la línea del azar cruzaba una de las etiquetas
    t.set_bbox(dict(facecolor="white", edgecolor="none", pad=1.5))
ax.axvline(90, color=TINTA, ls="--", lw=1.2)
ax.text(92, 0.04, "azar = 90°", transform=ax.get_xaxis_transform(),
        fontsize=9, color=TINTA)
ax.set_xlabel("Desviación mediana frente al desplazamiento real (grados)")
ax.set_xlim(0, 260)          # holgura para que quepa la etiqueta de la barra larga
ax.set_xticks(range(0, 181, 45))
ax.set_title("El identificador de parada que publica el feed apunta al sentido contrario",
             loc="left", fontsize=12)
ax.invert_yaxis()
fig.tight_layout(); fig.savefig("f4_rumbo.png"); plt.close(fig)
print("F4 ·", {m[0].replace(chr(10), " "): f"{m[1]:.1f}° (n={m[2]})" for m in med})

## 5. Predicciones negativas

**Datos:** el mismo CSV, filtrando `retraso_predicho_s` menor que cero.

Llegar antes de la hora oficial. 28 de 548, con un caso de −30 minutos detectado en
producción el 14/09 que además se mostraba con la etiqueta «En hora».


In [ ]:
# --- F5: predicciones negativas ----------------------------------------------
neg = df[df["pred_min"] < 0]
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.scatter(df["horizonte_min"], df["pred_min"], s=14, color=GRIS, alpha=.5,
           label=f"predicciones ({len(df)})")
ax.scatter(neg["horizonte_min"], neg["pred_min"], s=26, color=ACENTO,
           label=f"negativas ({len(neg)})")
ax.axhline(0, color=TINTA, lw=1.2)
ax.set_xlabel("Horizonte (minutos)"); ax.set_ylabel("Retraso predicho (minutos)")
ax.set_title("Predicciones negativas: llegar antes de la hora oficial", loc="left")
ax.legend(frameon=False, loc="lower right")
fig.tight_layout(); fig.savefig("f5_negativas.png"); plt.close(fig)

print("MAE modelo %.2f · constante 4 %.2f" % (valores[0], valores[1]))
print("r global %.3f" % df["pred_min"].corr(df["real_min"]))
print("figuras generadas")

## 6. Validación del 15/09, tras el filtro de dominio

**Datos:** `verificacion_15sep.csv`, generado con el mismo procedimiento que el del
13/09. Martes, de 15:42 a 18:37 UTC, una ronda cada 5 minutos y solo el instante
presente, como la aplicación.

Cada panel compara predictores **sobre la misma muestra**. No se comparan los errores de
los dos días entre sí: cambian el día de la semana, el nivel de retraso y el modelo.


In [ ]:
# --- F6: validacion del 15/09 -------------------------------------------------
d2 = pd.read_csv("verificacion_15sep.csv")
d2 = d2[(d2["estado"] == "ok") & (d2["via"] == "parada")].copy()
d2["pred_min"] = d2["retraso_predicho_s"] / 60
d2["real_min"] = d2["retraso_real_s"] / 60
mae2 = lambda p: float(np.abs(p - d2["real_min"]).mean())

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [1, 1.3]})
etq = ["Modelo", "Siempre\n4 min", "Horario\noficial"]
val = [mae2(d2["pred_min"]), mae2(4.0), mae2(0.0)]
b = a1.bar(etq, val, color=[ACENTO, GRIS, GRIS])
a1.bar_label(b, labels=[f"{coma(v, 2)} min" for v in val], padding=3, fontweight="bold")
a1.set_ylim(0, max(val) * 1.2)
a1.set_ylabel("Error absoluto medio (minutos)")
a1.yaxis.set_major_formatter(FMT)
a1.set_title("El modelo supera a las dos reglas", loc="left")

bins = np.arange(-2, 32, 1)
a2.hist(d2["real_min"], bins=bins, color=ACENTO, alpha=.55,
        label=f"observado · desv. típica {coma(d2['real_min'].std())} min")
a2.hist(d2["pred_min"], bins=bins, color=TINTA, alpha=.65,
        label=f"predicho · desv. típica {coma(d2['pred_min'].std())} min")
a2.set_xlabel("Retraso (minutos)"); a2.set_ylabel("Número de predicciones")
a2.xaxis.set_major_formatter(FMT)
a2.set_title("Pero sigue sin anticipar los retrasos altos", loc="left")
a2.legend(frameon=False)

fig.tight_layout()
fig.text(0.01, -0.03, f"15/09/2026 · {len(d2)} predicciones de {d2['nucleo'].nunique()} trenes · "
         "vía directa · mismos seis trayectos que el 13/09", fontsize=9, color=GRIS)
fig.savefig("f6_validacion_15sep.png"); plt.close(fig)
print(f"15/09 · n={len(d2)} · trenes={d2['nucleo'].nunique()} · modelo {val[0]:.2f} · "
      f"4 min {val[1]:.2f} · horario {val[2]:.2f} · r {d2['pred_min'].corr(d2['real_min']):.3f}")


## Comprobación de las cifras del anexo

Estas son las cifras que aparecen en el texto. Si alguna no coincide, manda el
cuaderno y hay que corregir el texto, no al revés.


In [ ]:
print(f"n                         : {len(df)}")
print(f"MAE modelo                : {np.abs(df['err_min']).mean():.2f} min")
print(f"mediana del error         : {np.abs(df['err_min']).median():.2f} min")
print(f"correlacion r             : {df['pred_min'].corr(df['real_min']):.3f}")
print(f"R2                        : {df['pred_min'].corr(df['real_min'])**2:.3f}")
print(f"sd observado / sd predicho: {df['real_min'].std() / df['pred_min'].std():.2f}x")
print(f"  excluyendo negativas    : ", end="")
pos = df[df["pred_min"] >= 0]
print(f"{pos['real_min'].std() / pos['pred_min'].std():.2f}x")
print(f"predicciones negativas    : {(df['pred_min'] < 0).sum()} ({100*(df['pred_min']<0).mean():.1f} %)")
print(f"MAE tras acotar a cero    : {np.abs(df['pred_min'].clip(lower=0) - df['real_min']).mean():.2f} min")